Merge Datasets Demografico + MIQ-3 + Dados Comportamentais

In [4]:
from pathlib import Path
import pandas as pd

# 1. Caminhos dos ficheiros
pasta_dados = Path("../data_subjects")

ficheiro_questionarios = (
    pasta_dados / "R_Base Dados_Questionários_Study 2_Guilherme Saldanha.xlsx"
)

ficheiro_comportamental = (
    pasta_dados / "Resultados_Comportamentais_N-54_Study 2 PhD.xlsx"
)

ficheiro_saida = pasta_dados / "Merge_Dataset_Study_2.xlsx"


# 2. Leitura dos ficheiros
# sheet_name=0 lê a primeira folha em cada ficheiro
df_questionarios = pd.read_excel(ficheiro_questionarios, sheet_name=0)
df_comportamental = pd.read_excel(ficheiro_comportamental, sheet_name=0)


# 3. Limpeza da variável-chave
df_questionarios.columns = df_questionarios.columns.str.strip()
df_comportamental.columns = df_comportamental.columns.str.strip()

df_questionarios["ID"] = (
    df_questionarios["ID"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df_comportamental["ID"] = (
    df_comportamental["ID"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# 4. Verificação antes do merge
duplicados_questionarios = df_questionarios["ID"].duplicated().sum()
duplicados_comportamental = df_comportamental["ID"].duplicated().sum()

print(f"Questionários: {df_questionarios.shape[0]} participantes")
print(f"Comportamental: {df_comportamental.shape[0]} participantes")
print(f"IDs duplicados nos questionários: {duplicados_questionarios}")
print(f"IDs duplicados nos dados comportamentais: {duplicados_comportamental}")

if duplicados_questionarios > 0 or duplicados_comportamental > 0:
    raise ValueError(
        "Existem IDs duplicados. Resolve-os antes de fazer o merge."
    )


# 5. Merge externo: mantém qualquer ID presente num dos ficheiros
merge_dataset = df_questionarios.merge(
    df_comportamental,
    on="ID",
    how="outer",
    validate="one_to_one",
    indicator="origem_merge"
)


# 6. Diagnóstico da correspondência de IDs
print("\nResultado do merge:")
print(merge_dataset["origem_merge"].value_counts())

ids_sem_correspondencia = merge_dataset.loc[
    merge_dataset["origem_merge"] != "both",
    ["ID", "origem_merge"]
]

if not ids_sem_correspondencia.empty:
    print("\nIDs sem correspondência entre os dois ficheiros:")
    display(ids_sem_correspondencia)
else:
    print("\nTodos os IDs aparecem nos dois ficheiros.")


# 7. Organizar por ID e exportar
merge_dataset = merge_dataset.sort_values("ID").reset_index(drop=True)

with pd.ExcelWriter(ficheiro_saida, engine="openpyxl") as writer:
    merge_dataset.to_excel(
        writer,
        sheet_name="Merge Dataset",
        index=False
    )

print(f"\nFicheiro criado com sucesso:\n{ficheiro_saida}")
print(f"Dimensões finais: {merge_dataset.shape}")

Questionários: 54 participantes
Comportamental: 54 participantes
IDs duplicados nos questionários: 0
IDs duplicados nos dados comportamentais: 0

Resultado do merge:
origem_merge
both          54
left_only      0
right_only     0
Name: count, dtype: int64

Todos os IDs aparecem nos dois ficheiros.

Ficheiro criado com sucesso:
..\data_subjects\Merge_Dataset_Study_2.xlsx
Dimensões finais: (54, 70)
